# FarrahAI — Notebook 3: Unsupervised Clustering

This notebook demonstrates:
- K-Means clustering of note chunks and questions
- Finding optimal k using elbow method + silhouette score
- Hierarchical clustering dendrogram
- 2D visualization using PCA/t-SNE
- Auto-discovery of hidden topic groups


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from modules.embedder import embed_texts
from modules.ml_models import cluster_topics, find_optimal_k

print('Ready ✓')

## Step 1: Load text data (note chunks or questions)

In [ ]:
# Use your actual extracted chunks or question paper text here.
# This synthetic set covers typical AI/ML exam topics.
texts = [
    "Backpropagation calculates gradients using chain rule",
    "Gradient descent updates weights in direction of negative gradient",
    "Learning rate controls step size during weight update",
    "Neural network activation functions include sigmoid relu tanh",
    "LSTM gates control what information to remember or forget",
    "Convolutional layer applies filters to detect local patterns",
    "Pooling reduces spatial dimensions while retaining important features",
    "Decision tree uses information gain to select best split attribute",
    "Gini impurity measures class mixing at a decision tree node",
    "Random forest aggregates predictions from many decision trees",
    "Bagging samples data with replacement for each tree",
    "XGBoost minimizes loss function using gradient boosting",
    "Boosting combines weak learners sequentially",
    "Support vector machine maximizes margin between class boundaries",
    "Kernel trick maps data to higher dimensional feature space",
    "K-means assigns each point to nearest centroid",
    "Silhouette score measures how similar a point is to its cluster",
    "DBSCAN groups points by density and detects outliers as noise",
    "Hierarchical clustering builds dendrogram by merging clusters",
    "PCA finds orthogonal directions of maximum variance",
    "Autoencoders compress input to latent space and reconstruct",
    "Variational autoencoder generates new samples from latent distribution",
    "Precision and recall trade off depending on classification threshold",
    "F1 score is harmonic mean of precision and recall",
    "ROC curve plots true positive rate versus false positive rate",
    "Cross validation provides unbiased model performance estimate",
    "Reinforcement learning agent maximizes long term cumulative reward",
    "Q learning estimates action value without model of environment",
    "Policy gradient methods optimize agent behavior directly",
    "Markov decision process defines state action reward transition",
]

print(f'Loaded {len(texts)} text samples')

## Step 2: Generate Embeddings

In [ ]:
print('Generating embeddings (downloads model on first run ~80MB)...')
embeddings = embed_texts(texts)
print(f'Embeddings shape: {embeddings.shape}')

## Step 3: Find Optimal k (Elbow + Silhouette)

In [ ]:
k_results = find_optimal_k(embeddings, k_range=range(2, 9))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_results['k_values'], k_results['inertias'], 'o-', color='#e74c3c')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(k_results['k_values'], k_results['silhouettes'], 'o-', color='#2ecc71')
axes[1].axvline(x=k_results['best_k'], color='gray', linestyle='--', label=f'Best k={k_results["best_k"]}')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/clustering_optimal_k.png', dpi=150)
plt.show()
print(f'Best k = {k_results["best_k"]}')

## Step 4: K-Means Clustering

In [ ]:
best_k = k_results['best_k']
cluster_df = cluster_topics(embeddings, texts, n_clusters=best_k, method='kmeans')

print(f'\nCluster assignments:')
for c in range(best_k):
    group = cluster_df[cluster_df['cluster'] == c]['text'].tolist()
    print(f'\n── Cluster {c} ({len(group)} items) ──────────────────')
    for t in group:
        print(f'  • {t}')

## Step 5: 2D Visualization (PCA)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings)

colors = plt.cm.Set2(np.linspace(0, 1, best_k))

fig, ax = plt.subplots(figsize=(11, 8))
for c in range(best_k):
    mask = cluster_df['cluster'] == c
    ax.scatter(coords[mask, 0], coords[mask, 1],
               color=colors[c], label=f'Cluster {c}', s=80, alpha=0.85)

# Annotate points
for i, txt in enumerate(texts):
    ax.annotate(txt[:30], (coords[i, 0], coords[i, 1]),
                fontsize=6.5, alpha=0.7, ha='center',
                xytext=(0, 6), textcoords='offset points')

ax.set_title(f'Topic Clusters (K-Means, k={best_k}) — PCA Projection')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/clustering_2d_pca.png', dpi=150)
plt.show()

## Step 6: Hierarchical Clustering Dendrogram

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(embeddings, method='ward')
short_labels = [t[:35] for t in texts]

fig, ax = plt.subplots(figsize=(14, 7))
dendrogram(Z, labels=short_labels, leaf_rotation=90, leaf_font_size=7, ax=ax)
ax.set_title('Hierarchical Clustering Dendrogram — Note Topics')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.savefig('../outputs/clustering_dendrogram.png', dpi=150)
plt.show()